In [1]:
!pip install -U transformers datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 47.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 61.9 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.4 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<

In [ ]:
# ============================================================
# SCAF MODEL — 5-SEED RETRAINING (Section 4.6)
# - Validation + Test results for each seed
# - Accuracy, Precision, Recall, F1 for each seed
# - Best model weights saved
# - Mean ± Std reported
# ============================================================

# ------------------------------------------------------------
# 1) Imports
# ------------------------------------------------------------
import time
import json
import os
import torch
import numpy as np
import pandas as pd
import torch.nn as nn

from transformers import (
    AutoTokenizer, AutoModel,
    TrainingArguments, Trainer
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ------------------------------------------------------------
# 2) CONFIG
# ------------------------------------------------------------
DATA_PATH    = "/kaggle/input/datasets/shuvo128/shuvoda/IMDB_Cleaned (1) (1).csv"
MODEL_NAME   = "google/electra-base-discriminator"
TFIDF_DIM    = 3000
SEEDS        = [42, 123, 456, 789, 1011]
RESULTS_FILE = "/kaggle/working/scaf_5seed_results.json"
SAVE_DIR     = "/kaggle/working/scaf_models"
os.makedirs(SAVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# 3) GPU Info
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU")

# ------------------------------------------------------------
# 4) Load Data — Fixed Split 40k/5k/5k
# ------------------------------------------------------------
print("\nLoading data...")
df = pd.read_csv(DATA_PATH)
text_column = "cleaned_review" if "cleaned_review" in df.columns else "review"
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

texts  = df[text_column].astype(str).tolist()
labels = (df["sentiment"] == "positive").astype(int).tolist()

train_texts  = texts[:40000];   train_labels  = labels[:40000]
val_texts    = texts[40000:45000]; val_labels = labels[40000:45000]
test_texts   = texts[45000:50000]; test_labels = labels[45000:50000]

print(f"Train: {len(train_texts)}, Val: {len(val_texts)}, Test: {len(test_texts)}")
assert len(train_texts) == 40000
assert len(val_texts)   == 5000
assert len(test_texts)  == 5000
print("✓ Split verified: 40k-5k-5k")

# ------------------------------------------------------------
# 5) TF-IDF (Fixed across seeds)
# ------------------------------------------------------------
print("\nFitting TF-IDF...")
tfidf = TfidfVectorizer(max_features=TFIDF_DIM)
tfidf.fit(train_texts)

train_tfidf = tfidf.transform(train_texts).toarray()
val_tfidf   = tfidf.transform(val_texts).toarray()
test_tfidf  = tfidf.transform(test_texts).toarray()
print("✓ TF-IDF ready")

# ------------------------------------------------------------
# 6) Tokenizer
# ------------------------------------------------------------
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts_list):
    return tokenizer(
        texts_list, truncation=True,
        padding=True, max_length=512
    )

print("Tokenizing...")
train_enc = tokenize(train_texts)
val_enc   = tokenize(val_texts)
test_enc  = tokenize(test_texts)
print("✓ Tokenization complete")

# ------------------------------------------------------------
# 7) Dataset Class
# ------------------------------------------------------------
class HybridDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, tfidf_features, labels):
        self.encodings = encodings
        self.tfidf     = tfidf_features
        self.labels    = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["tfidf"]  = torch.tensor(self.tfidf[idx], dtype=torch.float)
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = HybridDataset(train_enc, train_tfidf, train_labels)
val_dataset   = HybridDataset(val_enc,   val_tfidf,   val_labels)
test_dataset  = HybridDataset(test_enc,  test_tfidf,  test_labels)

# ------------------------------------------------------------
# 8) SCAF Model Definition
# ------------------------------------------------------------
class CrossAttention(nn.Module):
    def __init__(self, query_dim, key_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.tfidf_proj = nn.Linear(key_dim, query_dim)
        self.attn = nn.MultiheadAttention(
            embed_dim   = query_dim,
            num_heads   = num_heads,
            dropout     = dropout,
            batch_first = True
        )
        self.norm    = nn.LayerNorm(query_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, cls_output, tfidf):
        tfidf_proj  = self.tfidf_proj(tfidf).unsqueeze(1)
        query       = cls_output.unsqueeze(1)
        attn_out, _ = self.attn(
            query=query, key=tfidf_proj, value=tfidf_proj
        )
        attn_out = attn_out.squeeze(1)
        return self.norm(cls_output + self.dropout(attn_out))


class ElectraCrossAttnTFIDF(nn.Module):
    def __init__(self, model_name, tfidf_dim, num_labels=2):
        super().__init__()
        self.electra    = AutoModel.from_pretrained(model_name)
        hidden_size     = self.electra.config.hidden_size
        self.cross_attn = CrossAttention(
            query_dim = hidden_size,
            key_dim   = tfidf_dim,
            num_heads = 8,
            dropout   = 0.1
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(512, 128),         nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128, num_labels)
        )

    def forward(self, input_ids, attention_mask, tfidf,
                labels=None, token_type_ids=None):
        outputs    = self.electra(
            input_ids      = input_ids,
            attention_mask = attention_mask,
            token_type_ids = token_type_ids
        )
        cls_output = outputs.last_hidden_state[:, 0, :].contiguous()
        fused      = self.cross_attn(cls_output, tfidf)
        logits     = self.classifier(fused)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

# ------------------------------------------------------------
# 9) Metrics
# ------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy" : round(float(acc), 4),
        "precision": round(float(precision), 4),
        "recall"   : round(float(recall), 4),
        "f1"       : round(float(f1), 4),
    }

# ------------------------------------------------------------
# 10) Resume Support
# ------------------------------------------------------------
all_results = []
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        all_results = json.load(f)
    completed_seeds = [r["seed"] for r in all_results]
    print(f"\nResuming — completed seeds: {completed_seeds}")
else:
    completed_seeds = []

# Track best model
best_acc  = 0.0
best_seed = None

# ------------------------------------------------------------
# 11) MAIN LOOP — 5 Seeds
# ------------------------------------------------------------
for seed in SEEDS:
    if seed in completed_seeds:
        print(f"\nSeed {seed} already done, skipping...")
        # Check if this was best
        for r in all_results:
            if r["seed"] == seed and r["test_accuracy"] > best_acc:
                best_acc  = r["test_accuracy"]
                best_seed = seed
        continue

    print(f"\n{'='*60}")
    print(f"TRAINING — SEED {seed}")
    print(f"{'='*60}")

    # Set seeds
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Fresh model
    model = ElectraCrossAttnTFIDF(MODEL_NAME, TFIDF_DIM)

    training_args = TrainingArguments(
        output_dir                  = f"./scaf_results_seed{seed}",
        learning_rate               = 2e-5,
        lr_scheduler_type           = "cosine",
        per_device_train_batch_size = 16,
        per_device_eval_batch_size  = 16,
        num_train_epochs            = 4,
        weight_decay                = 0.01,
        max_grad_norm               = 1.0,
        warmup_steps                = 500,
        eval_strategy               = "epoch",
        save_strategy               = "no",
        load_best_model_at_end      = False,
        fp16                        = True,
        logging_steps               = 200,
        report_to                   = "none",
        seed                        = seed,
    )

    trainer = Trainer(
        model           = model,
        args            = training_args,
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,
        compute_metrics = compute_metrics,
    )

    # Train
    start_time = time.time()
    trainer.train()
    train_time = (time.time() - start_time) / 60

    # ── Validation Results ──
    print(f"\n--- Seed {seed} Validation Results ---")
    val_output  = trainer.evaluate(val_dataset)
    val_acc     = round(val_output["eval_accuracy"],  4)
    val_prec    = round(val_output["eval_precision"], 4)
    val_rec     = round(val_output["eval_recall"],    4)
    val_f1      = round(val_output["eval_f1"],        4)

    print(f"Val Accuracy : {val_acc}")
    print(f"Val Precision: {val_prec}")
    print(f"Val Recall   : {val_rec}")
    print(f"Val F1       : {val_f1}")

    # ── Test Results ──
    print(f"\n--- Seed {seed} Test Results ---")
    test_output = trainer.predict(test_dataset)
    test_preds  = np.argmax(test_output.predictions, axis=1)

    test_prec, test_rec, test_f1, _ = precision_recall_fscore_support(
        test_labels, test_preds, average='binary'
    )
    test_acc = accuracy_score(test_labels, test_preds)

    print(f"Test Accuracy : {round(test_acc, 4)}")
    print(f"Test Precision: {round(float(test_prec), 4)}")
    print(f"Test Recall   : {round(float(test_rec), 4)}")
    print(f"Test F1       : {round(float(test_f1), 4)}")
    print(f"Training Time : {train_time:.1f} min")

    # ── Save Model Weights ──
    model_save_path = os.path.join(SAVE_DIR, f"scaf_seed{seed}.pt")
    torch.save(model.state_dict(), model_save_path)
    tokenizer.save_pretrained(os.path.join(SAVE_DIR, f"tokenizer_seed{seed}"))
    print(f"✓ Model saved: {model_save_path}")

    # ── Track Best Model ──
    if test_acc > best_acc:
        best_acc  = test_acc
        best_seed = seed
        # Save as best model
        torch.save(
            model.state_dict(),
            os.path.join(SAVE_DIR, "scaf_best_model.pt")
        )
        tokenizer.save_pretrained(os.path.join(SAVE_DIR, "tokenizer_best"))
        print(f"✓ New best model saved! (seed {seed}, acc {best_acc:.4f})")

    # ── Save Result ──
    result = {
        "seed"          : seed,
        "val_accuracy"  : round(float(val_acc),      4),
        "val_precision" : round(float(val_prec),      4),
        "val_recall"    : round(float(val_rec),       4),
        "val_f1"        : round(float(val_f1),        4),
        "test_accuracy" : round(float(test_acc),      4),
        "test_precision": round(float(test_prec),     4),
        "test_recall"   : round(float(test_rec),      4),
        "test_f1"       : round(float(test_f1),       4),
        "train_time_min": round(float(train_time),    1),
        "model_path"    : model_save_path,
    }
    all_results.append(result)

    # Save JSON after every seed
    with open(RESULTS_FILE, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"✓ Results saved to JSON")

    # Free GPU memory
    del model, trainer
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 12) FINAL SUMMARY
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("SCAF — 5-SEED COMPLETE RESULTS")
print("=" * 60)

print(f"\n{'Seed':>6} | {'Val Acc':>8} | {'Val F1':>7} | {'Test Acc':>9} | {'Test Prec':>10} | {'Test Rec':>9} | {'Test F1':>8} | {'Time':>6}")
print("-" * 80)

for r in all_results:
    print(
        f"{r['seed']:>6} | "
        f"{r['val_accuracy']:>8.4f} | "
        f"{r['val_f1']:>7.4f} | "
        f"{r['test_accuracy']:>9.4f} | "
        f"{r['test_precision']:>10.4f} | "
        f"{r['test_recall']:>9.4f} | "
        f"{r['test_f1']:>8.4f} | "
        f"{r['train_time_min']:>5.1f}m"
    )

print("-" * 80)

# Mean ± Std
test_accs  = [r["test_accuracy"]  for r in all_results]
test_precs = [r["test_precision"] for r in all_results]
test_recs  = [r["test_recall"]    for r in all_results]
test_f1s   = [r["test_f1"]        for r in all_results]

print(f"\nMean Test Accuracy : {np.mean(test_accs):.4f} ± {np.std(test_accs):.4f}")
print(f"Mean Test Precision: {np.mean(test_precs):.4f} ± {np.std(test_precs):.4f}")
print(f"Mean Test Recall   : {np.mean(test_recs):.4f} ± {np.std(test_recs):.4f}")
print(f"Mean Test F1       : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")

print(f"\nReport as: {np.mean(test_accs)*100:.2f}% ± {np.std(test_accs)*100:.2f}%")
print(f"Best Seed : {best_seed} (Test Acc: {best_acc:.4f})")
print("=" * 60)
print(f"\nAll results saved: {RESULTS_FILE}")
print(f"All models saved : {SAVE_DIR}/")

GPU: Tesla T4

Loading data...
Train: 40000, Val: 5000, Test: 5000
✓ Split verified: 40k-5k-5k

Fitting TF-IDF...
✓ TF-IDF ready

Loading tokenizer...


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Test Accuracy : 0.953
Test Precision: 0.9485
Test Recall   : 0.9587
Test F1       : 0.9536
Training Time : 145.2 min
✓ Model saved: /kaggle/working/scaf_models/scaf_seed789.pt
✓ Results saved to JSON

TRAINING — SEED 1011


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
electra.embeddings_project.bias                   | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
electra.embeddings_project.weight                 | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # 

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.157629,0.157193,0.946400,0.952800,0.938200,0.945500
2,0.116925,0.174274,0.946600,0.933300,0.960800,0.946900
3,0.059709,0.216864,0.949600,0.942300,0.956800,0.949500


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [7]:
!find /kaggle -name "scaf_*"

In [8]:
!find /kaggle -name "*.pt"

In [5]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    print(root)
    for f in files:
        print("   ", f)

/kaggle/working
/kaggle/working/.virtual_documents
    __notebook_source__.ipynb


In [6]:
!find /kaggle -name "*.json"